In [28]:
from IPython.display import display, HTML
display(HTML("<style>.container { width:100% !important; }</style>"))

# Lab | Natural Language Processing
### SMS: SPAM or HAM

### Let's prepare the environment

In [29]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer

- Read Data for the Fraudulent Email Kaggle Challenge
- Reduce the training set to speead up development. 

In [30]:
## Read Data for the Fraudulent Email Kaggle Challenge
data = pd.read_csv("../data/kg_train.csv",encoding='latin-1')

# Reduce the training set to speed up development. 
# Modify for final system
data = data.head(1000)
print(data.shape)
data.fillna("",inplace=True)

(1000, 2)


,text,label
0,"DEAR SIR, STRICTLY A PRIVATE BUSINESS PROPOSAL...",1
1,Will do.,0
2,Nora--Cheryl has emailed dozens of memos about...,0
3,Dear Sir=2FMadam=2C I know that this proposal ...,1
4,fyi,0
...,...,...
995,So what's the latest? It sounds contradictory ...,0
996,"TRANSFER OF 36,759,000.00 MILLION POUNDS TO YO...",1
997,Barb I will call to explain. Are you back in t...,0
998,Yang on travelNot free tonite.May work tomorrow,0


### Let's divide the training and test set into two partitions

In [31]:
from sklearn.model_selection import train_test_split

data_train, data_val = train_test_split(data, test_size=0.2, random_state=42)
print(f"Train size: {data_train.shape}, Val size: {data_val.shape}")

Train size: (800, 2), Val size: (200, 2)


## Data Preprocessing

In [32]:
import string
from nltk.corpus import stopwords
print(string.punctuation)
print(stopwords.words("english")[100:110])
from nltk.stem.snowball import SnowballStemmer
snowball = SnowballStemmer('english')

!"#$%&'()*+,-./:;<=>?@[\]^_`{|}~
['needn', "needn't", 'no', 'nor', 'not', 'now', 'o', 'of', 'off', 'on']


## Now, we have to clean the html code removing words

- First we remove inline JavaScript/CSS
- Then we remove html comments. This has to be done before removing regular tags since comments can contain '>' characters
- Next we can remove the remaining tags

In [33]:
import re

def clean_html(text):
    # Remove inline JavaScript/CSS
    text = re.sub(r'<(script|style).*?>.*?</\1>', ' ', text, flags=re.DOTALL)
    # Remove HTML comments (before removing regular tags)
    text = re.sub(r'<!--.*?-->', ' ', text, flags=re.DOTALL)
    # Remove remaining HTML tags
    text = re.sub(r'<.*?>', ' ', text, flags=re.DOTALL)
    return text

data_train['clean_text'] = data_train['text'].apply(clean_html)
data_val['clean_text'] = data_val['text'].apply(clean_html)
data_train[['text', 'clean_text']].head(2)

,text,clean_text
29,"----------- REGARDS, MR NELSON SMITH.KINDLY RE...","----------- REGARDS, MR NELSON SMITH.KINDLY RE..."
535,I have not been able to reach oscar this am. W...,I have not been able to reach oscar this am. W...


- Remove all the special characters
    
- Remove numbers
    
- Remove all single characters
 
- Remove single characters from the start

- Substitute multiple spaces with single space

- Remove prefixed 'b'

- Convert to Lowercase

In [34]:
def preprocess_text(text):
    # Remove all special characters
    text = re.sub(r'\W', ' ', text)
    # Remove numbers
    text = re.sub(r'\d+', ' ', text)
    # Remove all single characters
    text = re.sub(r'\b\w\b', ' ', text)
    # Remove single characters from the start
    text = re.sub(r'^.\s+', ' ', text)
    # Substitute multiple spaces with single space
    text = re.sub(r'\s+', ' ', text)
    # Remove prefixed 'b'
    text = re.sub(r'^b\s+', '', text)
    # Convert to lowercase
    text = text.lower().strip()
    return text

data_train['preprocessed_text'] = data_train['clean_text'].apply(preprocess_text)
data_val['preprocessed_text'] = data_val['clean_text'].apply(preprocess_text)
data_train[['clean_text', 'preprocessed_text']].head(2)

,clean_text,preprocessed_text
29,"----------- REGARDS, MR NELSON SMITH.KINDLY RE...",regards mr nelson smith kindly reply me on my ...
535,I have not been able to reach oscar this am. W...,have not been able to reach oscar this am we a...


## Now let's work on removing stopwords
Remove the stopwords.

In [35]:
stop_words = set(stopwords.words('english'))

def remove_stopwords(text):
    return ' '.join([w for w in text.split() if w not in stop_words])

data_train['preprocessed_text'] = data_train['preprocessed_text'].apply(remove_stopwords)
data_val['preprocessed_text'] = data_val['preprocessed_text'].apply(remove_stopwords)
data_train['preprocessed_text'].head(3)

29     regards mr nelson smith kindly reply private e...
535           able reach oscar supposed send pdb receive
695    huma abedin checking pat work jack jake rest a...
Name: preprocessed_text, dtype: str

## Tame Your Text with Lemmatization
Break sentences into words, then use lemmatization to reduce them to their base form (e.g., "running" becomes "run"). See how this creates cleaner data for analysis!

In [36]:
import nltk
nltk.download('wordnet', quiet=True)
from nltk.stem import WordNetLemmatizer

lemmatizer = WordNetLemmatizer()

def lemmatize_text(text):
    return ' '.join([lemmatizer.lemmatize(word) for word in text.split()])

data_train['preprocessed_text'] = data_train['preprocessed_text'].apply(lemmatize_text)
data_val['preprocessed_text'] = data_val['preprocessed_text'].apply(lemmatize_text)
data_train['preprocessed_text'].head(3)

29     regard mr nelson smith kindly reply private em...
535           able reach oscar supposed send pdb receive
695    huma abedin checking pat work jack jake rest a...
Name: preprocessed_text, dtype: str

## Bag Of Words
Let's get the 10 top words in ham and spam messages (**EXPLORATORY DATA ANALYSIS**)

In [37]:
from collections import Counter

def get_top_words(df, label, n=10):
    texts = df[df['label'] == label]['preprocessed_text']
    all_words = ' '.join(texts).split()
    return Counter(all_words).most_common(n)

print("Top 10 HAM words:")
for word, count in get_top_words(data_train, 0):
    print(f"  {word}: {count}")

print("\nTop 10 SPAM words:")
for word, count in get_top_words(data_train, 1):
    print(f"  {word}: {count}")

Top 10 HAM words:
  state: 117
  pm: 97
  would: 94
  president: 89
  mr: 89
  time: 81
  percent: 80
  obama: 77
  call: 74
  secretary: 74

Top 10 SPAM words:
  money: 847
  account: 743
  bank: 646
  fund: 626
  u: 600
  transaction: 471
  business: 424
  mr: 422
  country: 422
  million: 370


## Extra features

In [38]:
# We add to the original dataframe two additional indicators (money symbols and suspicious words).
money_simbol_list = "|".join(["euro","dollar","pound","€",r"\$"])
suspicious_words = "|".join(["free","cheap","sex","money","account","bank","fund","transfer","transaction","win","deposit","password"])

data_train['money_mark'] = data_train['preprocessed_text'].str.contains(money_simbol_list)*1
data_train['suspicious_words'] = data_train['preprocessed_text'].str.contains(suspicious_words)*1
data_train['text_len'] = data_train['preprocessed_text'].apply(lambda x: len(x)) 

data_val['money_mark'] = data_val['preprocessed_text'].str.contains(money_simbol_list)*1
data_val['suspicious_words'] = data_val['preprocessed_text'].str.contains(suspicious_words)*1
data_val['text_len'] = data_val['preprocessed_text'].apply(lambda x: len(x)) 

data_train.head()

,text,label,clean_text,preprocessed_text,money_mark,suspicious_words,text_len
29,"----------- REGARDS, MR NELSON SMITH.KINDLY RE...",1,"----------- REGARDS, MR NELSON SMITH.KINDLY RE...",regard mr nelson smith kindly reply private em...,0,0,79
535,I have not been able to reach oscar this am. W...,0,I have not been able to reach oscar this am. W...,able reach oscar supposed send pdb receive,0,0,42
695,; Huma Abedin B6I'm checking with Pat on the 5...,0,; Huma Abedin B6I'm checking with Pat on the 5...,huma abedin checking pat work jack jake rest a...,0,0,76
557,I can have it announced here on Monday - can't...,0,I can have it announced here on Monday - can't...,announced monday today,0,0,22
836,BANK OF AFRICAAGENCE SAN PEDRO14 BP 1210 S...,1,BANK OF AFRICAAGENCE SAN PEDRO14 BP 1210 S...,bank africaagence san pedro bp san pedro cote ...,1,1,1102


## How would work the Bag of Words with Count Vectorizer concept?

In [39]:
from sklearn.feature_extraction.text import CountVectorizer

count_vec = CountVectorizer(max_features=5000)
X_train_bow = count_vec.fit_transform(data_train['preprocessed_text'])
X_val_bow = count_vec.transform(data_val['preprocessed_text'])

print("BoW - Train shape:", X_train_bow.shape)
print("BoW - Val shape:  ", X_val_bow.shape)

BoW - Train shape: (800, 5000)
BoW - Val shape:   (200, 5000)


## TF-IDF

- Load the vectorizer

- Vectorize all dataset

- print the shape of the vetorized dataset

In [40]:
tfidf = TfidfVectorizer(max_features=5000)
X_train_tfidf = tfidf.fit_transform(data_train['preprocessed_text'])
X_val_tfidf = tfidf.transform(data_val['preprocessed_text'])

print("TF-IDF - Train shape:", X_train_tfidf.shape)
print("TF-IDF - Val shape:  ", X_val_tfidf.shape)

TF-IDF - Train shape: (800, 5000)
TF-IDF - Val shape:   (200, 5000)


## And the Train a Classifier?

In [41]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report

y_train = data_train['label']
y_val = data_val['label']

nb = MultinomialNB()
nb.fit(X_train_tfidf, y_train)
y_pred = nb.predict(X_val_tfidf)

print("=== MultinomialNB with TF-IDF ===")
print(classification_report(y_val, y_pred, target_names=['HAM', 'SPAM']))

=== MultinomialNB with TF-IDF ===
              precision    recall  f1-score   support

         HAM       0.99      0.96      0.98       125
        SPAM       0.94      0.99      0.96        75

    accuracy                           0.97       200
   macro avg       0.96      0.97      0.97       200
weighted avg       0.97      0.97      0.97       200



### Extra Task - Implement a SPAM/HAM classifier

https://www.kaggle.com/t/b384e34013d54d238490103bc3c360ce

The classifier can not be changed!!! It must be the MultinimialNB with default parameters!

Your task is to **find the most relevant features**.

For example, you can test the following options and check which of them performs better:
- Using "Bag of Words" only
- Using "TF-IDF" only
- Bag of Words + extra flags (money_mark, suspicious_words, text_len)
- TF-IDF + extra flags


You can work with teams of two persons (recommended).

In [42]:
import scipy.sparse as sp

def evaluate(X_tr, X_v, y_tr, y_v, label=""):
    clf = MultinomialNB()
    clf.fit(X_tr, y_tr)
    y_p = clf.predict(X_v)
    print(f"=== {label} ===")
    print(classification_report(y_v, y_p, target_names=['HAM', 'SPAM']))

extra_train = data_train[['money_mark', 'suspicious_words', 'text_len']].values
extra_val   = data_val[['money_mark', 'suspicious_words', 'text_len']].values

evaluate(X_train_bow,   X_val_bow,   y_train, y_val, "BoW only")
evaluate(X_train_tfidf, X_val_tfidf, y_train, y_val, "TF-IDF only")
evaluate(sp.hstack([X_train_bow,   extra_train]), sp.hstack([X_val_bow,   extra_val]), y_train, y_val, "BoW + extra features")
evaluate(sp.hstack([X_train_tfidf, extra_train]), sp.hstack([X_val_tfidf, extra_val]), y_train, y_val, "TF-IDF + extra features")

=== BoW only ===
              precision    recall  f1-score   support

         HAM       0.98      0.98      0.98       125
        SPAM       0.96      0.97      0.97        75

    accuracy                           0.97       200
   macro avg       0.97      0.97      0.97       200
weighted avg       0.98      0.97      0.98       200

=== TF-IDF only ===
              precision    recall  f1-score   support

         HAM       0.99      0.96      0.98       125
        SPAM       0.94      0.99      0.96        75

    accuracy                           0.97       200
   macro avg       0.96      0.97      0.97       200
weighted avg       0.97      0.97      0.97       200

=== BoW + extra features ===
              precision    recall  f1-score   support

         HAM       0.98      0.96      0.97       125
        SPAM       0.94      0.97      0.95        75

    accuracy                           0.96       200
   macro avg       0.96      0.97      0.96       200
weighted